In [6]:
from pathlib import Path
import torch
import torch.nn as nn
import copy
import numpy as np

from evaluation import evaluate_model
from data_utils import seed_everything, build_datasets
from Models.AF_mamba import AFMamba

DATA_PATH = Path("Data/structured_dataset_1hz.pt")
FOLD_PATH = Path("Data/subject_folds.pt")

BATCH_SIZE = 16
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 10
MIN_EPOCHS = 30
NUM_EPOCHS = 1000
INPUT_SIZE = 3600
PREDICTION_HORIZON = 3600
SEED = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(SEED)

data = torch.load(DATA_PATH, weights_only=False)
fold_data = torch.load(FOLD_PATH, weights_only=False)
af_sets, nsr_sets = fold_data["af_sets"], fold_data["nsr_sets"]

print("Device:", device)
print("Subjects:", len(data))

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0

    for x, y in loader:
        x, y = x.to(device).float(), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    return running_loss / len(loader)


def validate_one_epoch(model, loader, criterion):
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device).float(), y.to(device)
            running_loss += criterion(model(x), y).item()

    return running_loss / len(loader)

all_val_metrics = []
all_test_metrics = []

for fold_idx in range(5):
    print(f"\n===== FOLD {fold_idx} =====")

    train_loader, val_loader, test_loader, *_ = build_datasets(data, af_sets, nsr_sets, fold_idx, input_segment_size=INPUT_SIZE, prediction_horizon=PREDICTION_HORIZON, batch_size=BATCH_SIZE)

    model = AFMamba().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    for epoch in range(NUM_EPOCHS):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss = validate_one_epoch(model, val_loader, criterion)

        print(f"Epoch {epoch + 1:04d} | " f"Train Loss={train_loss:.4f} | " f"Val Loss={val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE and epoch + 1 >= MIN_EPOCHS:
            print("Early stopping.")
            break

    model.load_state_dict(best_state)

    val_metrics, *_ = evaluate_model(model, val_loader, device=device, threshold=None)
    threshold = val_metrics["threshold"]

    test_metrics, *_ = evaluate_model(model, test_loader, device=device, threshold=threshold)

    print(f"\nBest Val Loss={best_val_loss:.4f}")
    print(f"Validation | " f"Sens={val_metrics['recall']:.4f}, " f"Spec={val_metrics['specificity']:.4f}, " f"F1={val_metrics['f1']:.4f}, " f"AUROC={val_metrics['roc_auc']:.4f}, " f"AUPRC={val_metrics['auprc']:.4f}")
    print(f"Test       | " f"Sens={test_metrics['recall']:.4f}, " f"Spec={test_metrics['specificity']:.4f}, " f"F1={test_metrics['f1']:.4f}, " f"AUROC={test_metrics['roc_auc']:.4f}, " f"AUPRC={test_metrics['auprc']:.4f}")

    all_val_metrics.append(val_metrics)
    all_test_metrics.append(test_metrics)

METRICS = {"Sensitivity": "recall", "Specificity": "specificity", "F1": "f1", "AUROC": "roc_auc", "AUPRC": "auprc"}

print("\n===== 5-FOLD TEST RESULTS =====")
for label, key in METRICS.items():
    values = np.array([m[key] for m in all_test_metrics], dtype=float)
    print(f"{label}: {values.mean():.4f} ± {values.std():.4f}")

Device: cuda
Subjects: 232

===== FOLD 0 =====

=== FOLD 0 ===
Subjects: train=140, val=46, test=46
Epoch 0001 | Train Loss=0.6876 | Val Loss=0.5404
Epoch 0002 | Train Loss=0.5972 | Val Loss=0.4341
Epoch 0003 | Train Loss=0.5254 | Val Loss=0.3574
Epoch 0004 | Train Loss=0.4762 | Val Loss=0.3218
Epoch 0005 | Train Loss=0.4360 | Val Loss=0.2865
Epoch 0006 | Train Loss=0.3964 | Val Loss=0.2784
Epoch 0007 | Train Loss=0.3601 | Val Loss=0.2714
Epoch 0008 | Train Loss=0.3469 | Val Loss=0.2785
Epoch 0009 | Train Loss=0.3203 | Val Loss=0.2748
Epoch 0010 | Train Loss=0.3069 | Val Loss=0.2815
Epoch 0011 | Train Loss=0.2793 | Val Loss=0.2678
Epoch 0012 | Train Loss=0.2649 | Val Loss=0.2711
Epoch 0013 | Train Loss=0.2524 | Val Loss=0.2741
Epoch 0014 | Train Loss=0.2529 | Val Loss=0.2543
Epoch 0015 | Train Loss=0.2245 | Val Loss=0.2579
Epoch 0016 | Train Loss=0.2248 | Val Loss=0.2488
Epoch 0017 | Train Loss=0.2062 | Val Loss=0.2588
Epoch 0018 | Train Loss=0.2058 | Val Loss=0.2491
Epoch 0019 | Train